# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atif929/flyrank-ml-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row represents the daily search performance of one content page. For this assignment, I will use data from March 2026 because it is a mid-panel month and avoids using the final month as a development dataset. My lane focuses on Content Refresh Opportunity Scoring, where each page is evaluated using its observed search performance over time.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Install packages

from huggingface_hub import hf_hub_download
from google.colab import userdata
import duckdb

# Get your HF token from Colab Secrets
token = userdata.get("HF_TOKEN")

# Download the March 2026 parquet file
file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=token
)

# Read with DuckDB
con = duckdb.connect()

df = con.execute(f"""
SELECT *
FROM read_parquet('{file_path}')
LIMIT 5
""").fetch_df()

print("Rows shown:", len(df))
print("Columns:", len(df.columns))
display(df)


Rows shown: 5
Columns: 31


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## Fields: feature / label / context / excluded

**Features:** clicks, impressions, CTR, average position, and page age. These are available before making a refresh decision and can help describe a page's search performance.

**Label (proxy):** Whether a page should be prioritized for content refresh. This is a defined rule based on observed search performance trends rather than a directly observed outcome.

**Context:** Date, page identifier, and content category. These provide additional information but are not the prediction target.

**Excluded:** Future performance data and any information created after the decision date are excluded to avoid data leakage and ensure the model only uses information available at the decision time.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Available columns:")
for col in df.columns:
    print("-", col)


Available columns:
- report_date
- client_hash_id
- content_hash_id
- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available
- gsc_impressions
- gsc_clicks
- gsc_sum_position
- gsc_avg_position
- ga4_pageviews
- ga4_sessions
- ga4_users
- ga4_engaged_sessions
- ga4_total_engagement_sec
- sessions_organic
- sessions_direct
- sessions_referral
- sessions_social
- sessions_paid
- sessions_ai
- ai_chatgpt
- ai_perplexity
- ai_gemini
- ai_copilot
- ai_claude
- ai_meta
- ai_other
- scroll_events
- month


## Verify it with queries

The following queries verify the data contract by checking the dataset grain, the number of records, missing values in important fields, and the date range for the selected month (March 2026). These checks confirm that the dataset matches the assumptions used for this ML task.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Verify grain
grain = con.execute(f"""
SELECT COUNT(*) AS total_rows,
COUNT(DISTINCT content_hash_id || report_date) AS unique_page_day
FROM read_parquet('{file_path}')
""").fetch_df()

print("=== Grain Check ===")
display(grain)

# 2. Row count and date window
summary = con.execute(f"""
SELECT
COUNT(*) AS rows,
MIN(report_date) AS start_date,
MAX(report_date) AS end_date
FROM read_parquet('{file_path}')
""").fetch_df()

print("=== Row Count & Date Window ===")
display(summary)

# 3. Missing values in main features
missing = con.execute(f"""
SELECT
SUM(gsc_clicks IS NULL) AS missing_clicks,
SUM(gsc_impressions IS NULL) AS missing_impressions,
SUM(gsc_avg_position IS NULL) AS missing_position
FROM read_parquet('{file_path}')
""").fetch_df()

print("=== Missing Values ===")
display(missing)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Grain Check ===


,total_rows,unique_page_day
0,9841378,9841378


=== Row Count & Date Window ===


,rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


=== Missing Values ===


,missing_clicks,missing_impressions,missing_position
0,0.0,0.0,6230317.0


## Data limits

This dataset supports decision-making but cannot prove why search performance changes. It contains observed search and engagement metrics rather than causal evidence. Some historical data may be incomplete because not every client has both Google Search Console and Google Analytics data available. The model also cannot account for external factors such as Google algorithm updates, competitor activity, or content changes that are not recorded in the dataset. Therefore, the results should be interpreted as decision-support rather than causal conclusions.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Data limitations documented.")


Data limitations documented.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.